# Reference copy - needs non-public data

이 노트북은 원본 연구 노트북의 **참고용 사본**입니다. 실행에 필요한 가격/시가총액/거래대금/ivol CSV(`./input/` 아래 6개 파일)은(는) 유료/비공개 데이터라 이 저장소에 포함하지 않았으므로 **그대로는 끝까지 실행되지 않습니다.** 출력, 실행 카운트, 로컬 경로는 제거했습니다. 실행 가능한 데모는 `00_public_sample_workflow.ipynb`(합성 데이터)를 사용하세요.

원본은 학회 세션 자료(단계별 폴더)였고, 파일 경로는 `./input/`, `./output/` 기준의 중립 경로로 바꿨습니다.

**공개 시 수정한 점:** (1) 리밸런싱 날짜를 `resample('QE')`(분기)에서 `resample('ME')`(월말)로 바꿨습니다. 본문과 저장된 `portfolio.csv`(309행, 월간)는 월간 기준입니다. (2) 거래대금 필터용 `threshold`와 Amihud 상위 20% 기준 `threshold`가 같은 변수명을 재사용하던 것을 `tv_threshold`, `illiq_threshold`로 분리했습니다. 로직(Amihud 기준은 `illiq.quantile(0.8)`)은 원본과 같습니다.

---
### **Portfolio formation**

* 수익률 극단치 조정: 상·하위 1% 윈저라이징
* 리밸런싱 주기: 월간 (2000-01-01 \~ 2025-08-31)
* 거래비용 반영:

  * Amihud 유동성 지표 상위 20% 종목: 80bp
  * 기타 종목: 30bp
* 유동성 필터링: 최근 60영업일 거래대금 하위 10% 종목 제외

---
##### **데이터 로드**

In [ ]:
import pandas as pd
import numpy as np

# 포트폴리오 수익률 계산을 위한 총수익률 데이터
total_adj_close  = pd.read_csv("./input/total adj close.csv", index_col=0, parse_dates=True)   # 현금배당 반영 수정주가(총수익률 계산)

# 포트폴리오 종목 선정을 위한 Investment factor 데이터
ivol             = pd.read_csv("./input/ivol.csv", index_col=0, parse_dates=True)              # Investment factor : ivol

# 시가총액가중 배분비중 계산 데이터
mkt_cap          = pd.read_csv("./input/mkt cap.csv", index_col=0, parse_dates=True)           # 시가총액

# 거래대금 하위 10% 필터링 계산 데이터
trading_value_60 = pd.read_csv("./input/trading value 60.csv", index_col=0, parse_dates=True)  # 60영업일 평균 거래대금

# Amihud illiquidity 계산 데이터
daily_ret        = pd.read_csv("./input/daily ret.csv", index_col=0, parse_dates=True)         # 일간 수익률 월별 데이터
trading_value    = pd.read_csv("./input/trading value.csv", index_col=0, parse_dates=True)     # 월말 거래대금

---
##### **총수익률 데이터 전처리**

In [ ]:
# 현금배당반영 수정주가 데이터를 이용해 월간 총수익률 계산
ret_m = total_adj_close.ffill().pct_change()

# 수익률 윈저라이징
ret_wins = ret_m.clip(
    lower=ret_m.quantile(0.01),
    upper=ret_m.quantile(0.99),
    axis=1
)

In [ ]:
ret_wins.tail()

---
##### **리밸런싱 날짜 설정**

In [ ]:
# 백테스트 기간을 설정하여 월말 날짜 리스트 생성
start_point = '1999-12-31'
end_point   = '2025-08-31'

month_ends = ivol.resample('ME').last().index
month_ends = month_ends[(month_ends >= pd.to_datetime(start_point)) &
                        (month_ends <= pd.to_datetime(end_point))]

month_ends

---
##### **포트폴리오 수익률 계산 - 1st Cycle**

Investment factor 설정

In [ ]:
factor_df = ivol

포트폴리오 설정

In [ ]:
n                = 20     # 종목 수
high_cost        = 0.008  # 80bp
low_cost         = 0.003  # 30bp

initial_NAV      = 1   # 초기값
NAV              = initial_NAV

portfolio_return = pd.Series(dtype=float)
total_trade      = pd.Series(dtype=float)

첫 Cycle 날짜 설정

In [ ]:
i = 0                               # 날짜 인덱스 설정
start_date = month_ends[i]          # 리밸런싱 기준일 (전월말)
end_date = month_ends[i + 1]        # 다음 리밸런싱 시점 (다음 월말)

거래대금 하위 10% 필터링

In [ ]:
series = trading_value_60.loc[start_date]
tv_threshold = series.quantile(0.1)   # 거래대금 하위 10% threshold
filtered = series[series > tv_threshold]

# filtered의 index만 남기기
factor_filtered = factor_df.loc[start_date, filtered.index]

포트폴리오 종목 선정

In [ ]:
basket = factor_filtered.nsmallest(n)      # 변동성 하위 20개 종목
# basket = factor_filtered.nlargest(n)      # 변동성 상위 20개 종목

Amihud illiquidity 계산

In [ ]:
illiq = (daily_ret.loc[start_date].abs() / trading_value.loc[start_date])

# 상위 20% (illiquid)
illiq_threshold = illiq.quantile(0.8)
illiquid_top20 = illiq[illiq >= illiq_threshold].index

거래비용 반영

In [ ]:
# 목표 자산배분 비중 계산

# 시총가중
# cap_seg        = mkt_cap.loc[start_date, basket.index]
# target_weights = (cap_seg / cap_seg.sum())

# 동일가중
target_weights = pd.Series(1/len(basket), index=basket.index, name=start_date)

In [ ]:
target_weights

In [ ]:
# 첫 거래 가정
prev_weights   = pd.Series(0, index=basket.index, name=start_date)

In [ ]:
prev_weights

In [ ]:
# 1. 비중 차이
all_index = target_weights.index.union(prev_weights.index)   # 전체 종목 universe = 두 시점 종목의 합집합

target_w = target_weights.reindex(all_index, fill_value=0)   # 비중 시리즈를 동일한 index로 확장
prev_w   = prev_weights.reindex(all_index, fill_value=0)     # 없으면 0으로 채움

delta_w = target_w - prev_w                                  # 차이 계산

In [ ]:
delta_w

In [ ]:
# 2. 거래금액 (매수/매도 절댓값)
trade_amounts = abs(delta_w) * NAV

# 3. 거래비용률 결정 (illiquid 종목은 높은 비용 적용)
cost_rate = np.where(delta_w.index.isin(illiquid_top20), high_cost, low_cost)  # 80bp vs 30bp

# 4. 총 거래비용
trade_cost = (trade_amounts * cost_rate).sum()

# 5. NAV 업데이트
NAV_new = NAV - trade_cost

# 6. 새 포트폴리오 가치
current_portfolio_value = target_weights * NAV_new

In [ ]:
current_portfolio_value

In [ ]:
ret_seg              = ret_wins.loc[start_date:end_date, basket.index].iloc[1:]

In [ ]:
next_portfolio_value = current_portfolio_value * (ret_seg + 1).cumprod()

In [ ]:
next_portfolio_value.sum(axis = 1)

In [ ]:
# 당기 포트폴리오 최종 가치 계산
ret_seg              = ret_wins.loc[end_date, basket.index]
next_portfolio_value = current_portfolio_value * (ret_seg + 1)

# 당기 포트폴리오 수익률 계산
NAV_new       = next_portfolio_value.sum()
portfolio_ret = NAV_new / NAV - 1

In [ ]:
print('포트폴리오 월초 NAV :', NAV)
print('포트폴리오 월말 NAV :', round(NAV_new, 3))
print('포트폴리오 월 수익률 :', round(portfolio_ret, 3))

In [ ]:
# prev_portfolio 업데이트
prev_portfolio = next_portfolio_value

# NAV 업데이트
NAV = NAV_new

# 총 거래금액 저장
total_trade.loc[start_date] = trade_amounts.sum()

# 포트폴리오 수익률 저장
portfolio_return.loc[end_date] = portfolio_ret

---
##### **전체 기간 포트폴리오 수익률 계산**

In [ ]:
for i in range(1, len(month_ends) - 1):

    # 날짜 업데이트
    start_date = month_ends[i]
    end_date = month_ends[i + 1] 

    # 거래대금 하위 10% 필터링
    series = trading_value_60.loc[start_date]
    tv_threshold = series.quantile(0.1)
    filtered = series[series > tv_threshold]
    factor_filtered = factor_df.loc[start_date, filtered.index]

    # 종목 선정
    basket = factor_filtered.nsmallest(n)      # 변동성 하위 20개 종목
    # basket = factor_filtered.nlargest(n)      # 변동성 상위 20개 종목

    # Amihud illiquidity 계산
    illiq = (daily_ret.loc[start_date].abs() / trading_value.loc[start_date])
    illiq_threshold = illiq.quantile(0.8)
    illiquid_top20 = illiq[illiq >= illiq_threshold].index

    # 자산배분 비중 계산
    prev_weights   = prev_portfolio / prev_portfolio.sum()
    # ## 시총가중
    # cap_seg        = mkt_cap.loc[start_date, basket.index]
    # target_weights = (cap_seg / cap_seg.sum())
    
    ### 동일가중
    target_weights = pd.Series(1/len(basket), index=basket.index, name=start_date)

    # 거래비용 반영
    all_index = target_weights.index.union(prev_weights.index)
    target_w = target_weights.reindex(all_index, fill_value=0)
    prev_w   = prev_weights.reindex(all_index, fill_value=0)
    delta_w = target_w - prev_w
    trade_amounts = abs(delta_w) * NAV
    cost_rate = np.where(delta_w.index.isin(illiquid_top20), high_cost, low_cost)
    trade_cost = (trade_amounts * cost_rate).sum()
    NAV_new = NAV - trade_cost
    current_portfolio_value = target_weights * NAV_new

    # 당기 포트폴리오 최종 가치 계산
    ret_seg              = ret_wins.loc[end_date, basket.index]
    next_portfolio_value = current_portfolio_value * (ret_seg + 1)
    
    # 당기 포트폴리오 수익률 계산
    NAV_new       = next_portfolio_value.sum()
    portfolio_ret = NAV_new / NAV - 1

    # prev_portfolio 업데이트
    prev_portfolio = next_portfolio_value

    # NAV 업데이트
    NAV = NAV_new

    # 총 거래금액 저장
    total_trade.loc[start_date] = trade_amounts.sum()
    
    # 포트폴리오 수익률 저장
    portfolio_return.loc[end_date] = portfolio_ret

In [ ]:
portfolio_return.tail()

---
##### **NAV 계산 및 OUTPUT 데이터 전처리**

In [ ]:
portfolio_NAV = (1 + portfolio_return).cumprod() * initial_NAV

In [ ]:
# Portfolio 데이터 생성
df = pd.concat(
    [portfolio_return, portfolio_NAV, total_trade], 
    axis=1
)

df.columns = ["Return", "NAV", "Trade"]
df.index.name = "Date"

# 초기값 저장
df.loc[df.index[0], "NAV"] = initial_NAV

In [ ]:
df

---
##### **데이터 저장**

In [ ]:
# Output 폴더
df.to_csv("./output/portfolio.csv", index=True, encoding="utf-8-sig")

# 다음 단계(03)에서 사용
# 03 노트북이 읽는 위치: ./input/portfolio.csv (원본에서는 다음 단계 폴더로 복사)